# Thử VGGT + Video Depth Anything trên video bảo tàng

Notebook chạy riêng trên `1.mp4`, `2.mp4`, `3.mp4`: Video Depth Anything (metric ViT-S) tạo depth liên tục, VGGT ước lượng camera/depth/point map từ 6 khung hình rải đều. Lưu file depth/3D và preview để đánh giá trước khi thay nhánh UniDepth. Đây là thử nghiệm hình học, chưa tự gắn nhãn touch.

Chọn Colab GPU. Độ sâu metric đơn camera vẫn có thể lệch scale trong bảo tàng; cần hiệu chuẩn bằng kích thước thật. VGGT checkpoint gốc có điều khoản non-commercial, hãy dùng theo giấy phép phù hợp.

In [ ]:
# Cài repo chính thức, giữ nguyên PyTorch/CUDA có sẵn của Colab.
!git clone -q https://github.com/DepthAnything/Video-Depth-Anything.git /content/Video-Depth-Anything
!git clone -q https://github.com/facebookresearch/vggt.git /content/vggt
!pip -q install timm einops huggingface_hub decord easydict imageio imageio-ffmpeg
!mkdir -p /content/Video-Depth-Anything/checkpoints
!wget -q -O /content/Video-Depth-Anything/checkpoints/metric_video_depth_anything_vits.pth https://huggingface.co/depth-anything/Metric-Video-Depth-Anything-Small/resolve/main/metric_video_depth_anything_vits.pth
!test -s /content/Video-Depth-Anything/checkpoints/metric_video_depth_anything_vits.pth && echo 'VDA metric checkpoint ready'

In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()  # upload 1.mp4, 2.mp4, 3.mp4
for name in ('1.mp4','2.mp4','3.mp4'):
    if name not in uploaded: raise FileNotFoundError(f'Missing {name}')
    Path('/content',name).write_bytes(uploaded[name])
print('3 videos uploaded')

In [ ]:
import sys,cv2,json,subprocess,numpy as np,torch
from pathlib import Path
from PIL import Image
sys.path[:0]=['/content/vggt','/content/Video-Depth-Anything']
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map
if not torch.cuda.is_available(): raise RuntimeError('Chọn GPU trong Colab Runtime settings.')
OUT=Path('/content/vggt_vda_results'); OUT.mkdir(exist_ok=True)
device='cuda'; dtype=torch.bfloat16 if torch.cuda.get_device_capability()[0]>=8 else torch.float16
model=VGGT.from_pretrained('facebook/VGGT-1B').to(device).eval()
print('VGGT loaded:',torch.cuda.get_device_name(0),dtype)

In [ ]:
def sample_frames(video,count=6,max_side=960):
    cap=cv2.VideoCapture(str(video)); n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps=float(cap.get(cv2.CAP_PROP_FPS))
    if n<1: raise RuntimeError(f'Cannot read {video}')
    ids=np.unique(np.linspace(0,n-1,min(count,n),dtype=int)); paths=[]
    for idx in ids:
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(idx)); ok,bgr=cap.read()
        if not ok: continue
        h,w=bgr.shape[:2]; scale=min(1.,max_side/max(h,w)); rgb=cv2.cvtColor(cv2.resize(bgr,(round(w*scale),round(h*scale))),cv2.COLOR_BGR2RGB)
        p=OUT/f'{video.stem}_frame_{idx:06d}.jpg'; Image.fromarray(rgb).save(p); paths.append((int(idx),str(p),rgb.shape[:2]))
    cap.release(); return fps,paths

def run_vda(video):
    out=OUT/'vda'; out.mkdir(exist_ok=True)
    cmd=['python','/content/Video-Depth-Anything/run.py','--input_video',str(video),'--output_dir',str(out),'--encoder','vits','--metric','--save_npz','--max_res','960']
    subprocess.run(cmd,check=True)
    return np.load(out/f'{video.stem}_depths.npz')['depths'].astype(np.float32)

def run_vggt(paths):
    imgs=load_and_preprocess_images([p[1] for p in paths]).to(device)
    with torch.inference_mode(),torch.autocast('cuda',dtype=dtype): pred=model(imgs)
    extr,intr=pose_encoding_to_extri_intri(pred['pose_enc'],imgs.shape[-2:])
    return {'vggt_depth':pred['depth'].float().cpu().numpy(),'vggt_depth_conf':pred['depth_conf'].float().cpu().numpy(),'vggt_world_points':pred['world_points'].float().cpu().numpy(),'vggt_world_points_conf':pred['world_points_conf'].float().cpu().numpy(),'extrinsic':extr.float().cpu().numpy(),'intrinsic':intr.float().cpu().numpy()}

reports=[]
for name in ('1.mp4','2.mp4','3.mp4'):
    video=Path('/content')/name; fps,views=sample_frames(video); print(name,':',len(views),'VGGT views')
    vda=run_vda(video); vg=run_vggt(views)
    np.savez_compressed(OUT/f'{video.stem}_geometry.npz',vda_depth=vda,vggt_frame_ids=np.asarray([x[0] for x in views]),**vg)
    # VDA metric depth visualization (colors normalized consistently per clip).
    finite=vda[np.isfinite(vda)&(vda>0)]; lo,hi=np.percentile(finite,[2,98]); cap=cv2.VideoCapture(str(video)); writer=None; fi=0
    while True:
        ok,bgr=cap.read()
        if not ok: break
        if fi>=len(vda): break
        d=cv2.resize(vda[fi],(bgr.shape[1],bgr.shape[0])); gray=np.clip((d-lo)/(hi-lo+1e-6),0,1)
        vis=cv2.applyColorMap((gray*255).astype(np.uint8),cv2.COLORMAP_TURBO)
        if writer is None: writer=cv2.VideoWriter(str(OUT/f'{video.stem}_vda_depth_preview.mp4'),cv2.VideoWriter_fourcc(*'mp4v'),max(fps,1),(vis.shape[1],vis.shape[0]))
        writer.write(vis); fi+=1
    cap.release()
    if writer: writer.release()
    reports.append({'video':name,'fps':fps,'vda_depth_shape':list(vda.shape),'vggt_views':len(views),'vggt_depth_shape':list(vg['vggt_depth'].shape),'vda_metric_depth_p02_p98':[float(lo),float(hi)]})
(OUT/'geometry_report.json').write_text(json.dumps(reports,indent=2)); print(json.dumps(reports,indent=2))

In [ ]:
from google.colab import files
import zipfile
archive='/content/vggt_vda_geometry_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(OUT))
files.download(archive)